# 1 langchain调用模型

In [5]:
# pip install -qU "langchain[openai]"
import os
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage # 大模型调用时候，role

In [26]:
# qwen 地址代替 openai,这是一个qwen大模型的key
os.environ["OPENAI_API_KEY"] = "sk-bfc69abc05ae4a91bd6e172fddaa9a35"
os.environ["OPENAI_BASE_URL"] ="https://dashscope.aliyuncs.com/compatible-mode/v1"

# 调用方式1
# 初始化客户端
chatLLM = ChatOpenAI(
    api_key="sk-bfc69abc05ae4a91bd6e172fddaa9a35",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "你是谁？"}
]
response = chatLLM.invoke(messages)
print(response.json())

# 调用方式2
model = init_chat_model("qwen-plus", model_provider="openai")
messages = [
    HumanMessage(content="你是谁？"),
]

print(model.invoke(messages))

/tmp/ipykernel_3996014/3753175831.py:17: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  print(response.json())


{"content":"我是Qwen，由阿里云开发的超大规模语言模型。我被设计用来回答问题、创作文字，比如写故事、写公文、写邮件、写剧本等等，还能表达观点，玩游戏等。我的目标是成为人类的好助手，帮助人们提高工作效率和生活质量。有什么我可以帮助你的吗？","additional_kwargs":{"refusal":null},"response_metadata":{"token_usage":{"completion_tokens":67,"prompt_tokens":22,"total_tokens":89,"completion_tokens_details":null,"prompt_tokens_details":{"audio_tokens":null,"cached_tokens":0}},"model_name":"qwen-max","system_fingerprint":null,"id":"chatcmpl-2f9e9e29-d13e-93bf-ba41-38e26795203e","service_tier":null,"finish_reason":"stop","logprobs":null},"type":"ai","name":null,"id":"run--2cebe1c5-19e9-4ba3-bfa2-b30af35c22dc-0","example":false,"tool_calls":[],"invalid_tool_calls":[],"usage_metadata":{"input_tokens":22,"output_tokens":67,"total_tokens":89,"input_token_details":{"cache_read":0},"output_token_details":{}}}
content='你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我可以回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_toke

# 4 pydantic定义tools

In [57]:
from pydantic import BaseModel, Field # 定义传入的数据请求格式
from typing import List
from typing_extensions import Literal

import openai
import json

client = openai.OpenAI(
    api_key="sk-bfc69abc05ae4a91bd6e172fddaa9a35", # https://bailian.console.aliyun.com/?tab=model#/api-key#我的支付宝申请的key
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
#传入的是我们函数的描述，然后让大模型选择传参
tools = [
    {
        "type": "function",
        "function": {
            "name": "Ticket",
            "description": "根据用户提供的信息查询火车时刻",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "description": "要查询的火车日期",
                        "title": "Date",
                        "type": "string",
                    },
                    "departure": {
                        "description": "出发城市或车站",
                        "title": "Departure",
                        "type": "string",
                    },
                    "destination": {
                        "description": "要查询的火车日期",
                        "title": "Destination",
                        "type": "string",
                    },
                },
                "required": ["date", "departure", "destination"],
            },
        },
    }
]

messages = [
    {
        "role": "user",
        "content": "你能帮我查一下2024年1月1日从北京南站到上海的火车票吗？"
    }
]

response = client.chat.completions.create(
    model="qwen-plus",
    messages=messages,
    tools=tools, # 生成函数的调用方式，并不是所有的模型都支持（某些比较小的模型不支持）
    tool_choice="auto",
)
print(response.choices[0].message.tool_calls[0].function)

Function(arguments='{"date": "2024年1月1日", "departure": "北京南站", "destination": "上海"}', name='Ticket')


In [59]:
response.choices[0].message.tool_calls[0].function.arguments

'{"date": "2024年1月1日", "departure": "北京南站", "destination": "上海"}'

In [21]:
############################老师的key已经失效###############################
# from pydantic import BaseModel, Field # 定义传入的数据请求格式
# from typing import List
# from typing_extensions import Literal

# import openai
# import json

# client = openai.OpenAI(
#     api_key="sk-78cc4e9ac8f44efdb207b7232e1ae6d8", # https://bailian.console.aliyun.com/?tab=model#/api-key
#     base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
# )
# #传入的是我们函数的描述，然后让大模型选择传参
# tools = [
#     {
#         "type": "function",
#         "function": {
#             "name": "Ticket",
#             "description": "根据用户提供的信息查询火车时刻",
#             "parameters": {
#                 "type": "object",
#                 "properties": {
#                     "date": {
#                         "description": "要查询的火车日期",
#                         "title": "Date",
#                         "type": "string",
#                     },
#                     "departure": {
#                         "description": "出发城市或车站",
#                         "title": "Departure",
#                         "type": "string",
#                     },
#                     "destination": {
#                         "description": "要查询的火车日期",
#                         "title": "Destination",
#                         "type": "string",
#                     },
#                 },
#                 "required": ["date", "departure", "destination"],
#             },
#         },
#     }
# ]

# messages = [
#     {
#         "role": "user",
#         "content": "你能帮我查一下2024年1月1日从北京南站到上海的火车票吗？"
#     }
# ]

# response = client.chat.completions.create(
#     model="qwen-plus",
#     messages=messages,
#     tools=tools, # 生成函数的调用方式，并不是所有的模型都支持（某些比较小的模型不支持）
#     tool_choice="auto",
# )
# print(response.choices[0].message.tool_calls[0].function)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided. For details, see: https://help.aliyun.com/zh/model-studio/error-code#apikey-error', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}, 'request_id': '7ef35aa8-da82-9005-acc7-3b2f1fde9146'}

In [49]:
client = openai.OpenAI(
    api_key="sk-bfc69abc05ae4a91bd6e172fddaa9a35", # https://bailian.console.aliyun.com/?tab=model#/api-key#我的支付宝申请的key
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
class ExtractionAgent:
   
    def __init__(self, model_name: str):
        self.model_name = model_name

    def call(self, user_prompt, response_model): #这里的response_model是指pydantic的Base_Model的子类，和大模型没有关系
        messages = [
            {
                "role": "user",
                "content": user_prompt
            }
        ]
        tools = [
            {
                "type": "function",
                "function": {
                    "name": response_model.model_json_schema()['title'], # 工具名字
                    "description": response_model.model_json_schema()['description'], # 工具描述
                    "parameters": {
                        "type": "object",
                        "properties": response_model.model_json_schema()['properties'], # 参数说明
                        "required": response_model.model_json_schema()['required'], # 必须要传的参数
                    },
                }
            }
        ]

        response = client.chat.completions.create(
            model=self.model_name,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        try:
            arguments = response.choices[0].message.tool_calls[0].function.arguments#获取定义pydantic模型的字段，然后让response_model.model_validate_json构建出这个pydantic类型的实例化
            return response_model.model_validate_json(arguments)
        except:
            print('ERROR', response.choices[0].message)
            return None


class Text(BaseModel):
    """文本问答内容解析"""
    search: bool = Field(description="是否需要搜索")
    keywords: List[str] = Field(description="待选关键词")
    intent: Literal['music', 'app', 'weather'] = Field(description="意图")
result = ExtractionAgent(model_name = "qwen-plus").call('汽车发动和轮胎出故障了，如何处理？', Text)
print(result)

search=True keywords=['汽车故障', '轮胎'] intent='app'


In [64]:
response_model.model_json_schema()['properties']

{'search': {'description': '是否需要搜索', 'title': 'Search', 'type': 'boolean'},
 'keywords': {'description': '待选关键词',
  'items': {'type': 'string'},
  'title': 'Keywords',
  'type': 'array'},
 'intent': {'description': '意图',
  'enum': ['music', 'app', 'weather'],
  'title': 'Intent',
  'type': 'string'}}

In [41]:
result

Text(search=True, keywords=['汽车故障', '轮胎'], intent='app')

In [30]:
#这里的response_model是指pydantic的Base_Model的子类，和大模型没有关系
#就是将Text这个类的结构提取出来
print(Text.model_json_schema())

{'description': '文本问答内容解析', 'properties': {'search': {'description': '是否需要搜索', 'title': 'Search', 'type': 'boolean'}, 'keywords': {'description': '待选关键词', 'items': {'type': 'string'}, 'title': 'Keywords', 'type': 'array'}, 'intent': {'description': '意图', 'enum': ['music', 'app', 'weather'], 'title': 'Intent', 'type': 'string'}}, 'required': ['search', 'keywords', 'intent'], 'title': 'Text', 'type': 'object'}


In [31]:
user_prompt = '汽车发动和轮胎出故障了，如何处理？'
messages = [
            {
                "role": "user",
                "content": user_prompt
            }
        ]

In [52]:
response_model = Text

In [35]:
tools = [
            {
                "type": "function",
                "function": {
                    "name": response_model.model_json_schema()['title'], # 工具名字
                    "description": response_model.model_json_schema()['description'], # 工具描述
                    "parameters": {
                        "type": "object",
                        "properties": response_model.model_json_schema()['properties'], # 参数说明
                        "required": response_model.model_json_schema()['required'], # 必须要传的参数
                    },
                }
            }
        ]

In [37]:
tools

[{'type': 'function',
  'function': {'name': 'Text',
   'description': '文本问答内容解析',
   'parameters': {'type': 'object',
    'properties': {'search': {'description': '是否需要搜索',
      'title': 'Search',
      'type': 'boolean'},
     'keywords': {'description': '待选关键词',
      'items': {'type': 'string'},
      'title': 'Keywords',
      'type': 'array'},
     'intent': {'description': '意图',
      'enum': ['music', 'app', 'weather'],
      'title': 'Intent',
      'type': 'string'}},
    'required': ['search', 'keywords', 'intent']}}}]

In [53]:
response = client.chat.completions.create(
            model='qwen-plus',
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

In [42]:
response

ChatCompletion(id='chatcmpl-a9e30a11-216e-9a3c-9e73-36fcff5783ac', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='汽车发动和轮胎出故障属于车辆维修问题，与火车时刻查询无关。建议您：\n\n1. **汽车无法启动**：\n   - 检查电瓶是否有电（如车灯是否亮、喇叭是否响）；\n   - 确认档位是否在P/N档（自动挡）或离合器是否踩到底（手动挡）；\n   - 尝试搭电或联系道路救援。\n\n2. **轮胎故障（如爆胎、漏气）**：\n   - 安全停车，开启双闪，放置三角警示牌；\n   - 若有备胎和工具，可自行更换；否则建议呼叫专业救援或轮胎服务。\n\n如需进一步帮助（例如查找附近维修点或救援电话），请告知您的具体位置或需求，我将尽力协助。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1773917339, model='qwen-plus', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=171, prompt_tokens=229, total_tokens=400, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0)))

In [43]:
response.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='汽车发动和轮胎出故障属于车辆维修问题，与火车时刻查询无关。建议您：\n\n1. **汽车无法启动**：\n   - 检查电瓶是否有电（如车灯是否亮、喇叭是否响）；\n   - 确认档位是否在P/N档（自动挡）或离合器是否踩到底（手动挡）；\n   - 尝试搭电或联系道路救援。\n\n2. **轮胎故障（如爆胎、漏气）**：\n   - 安全停车，开启双闪，放置三角警示牌；\n   - 若有备胎和工具，可自行更换；否则建议呼叫专业救援或轮胎服务。\n\n如需进一步帮助（例如查找附近维修点或救援电话），请告知您的具体位置或需求，我将尽力协助。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))

In [50]:
response.choices[0].message

ChatCompletionMessage(content='汽车发动和轮胎出故障属于车辆维修问题，与火车时刻查询无关。建议您：\n\n1. **汽车无法启动**：\n   - 检查电瓶是否有电（如车灯是否亮、喇叭是否响）；\n   - 确认档位是否在P/N档（自动挡）或离合器是否踩到底（手动挡）；\n   - 尝试搭电或联系道路救援。\n\n2. **轮胎故障（如爆胎、漏气）**：\n   - 安全停车，开启双闪，放置三角警示牌；\n   - 若有备胎和工具，可自行更换；否则建议呼叫专业救援或轮胎服务。\n\n如需进一步帮助（例如查找附近维修点或救援电话），请告知您的具体位置或需求，我将尽力协助。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)

In [54]:
response.choices[0].message.tool_calls[0]

ChatCompletionMessageFunctionToolCall(id='call_7eef68688a964025b977c5', function=Function(arguments='{"intent": "app", "keywords": ["汽车故障", "轮胎"], "search": true}', name='Text'), type='function', index=0)

In [55]:
response.choices[0].message.tool_calls[0]

ChatCompletionMessageFunctionToolCall(id='call_7eef68688a964025b977c5', function=Function(arguments='{"intent": "app", "keywords": ["汽车故障", "轮胎"], "search": true}', name='Text'), type='function', index=0)

In [56]:
response.choices[0].message.tool_calls[0].function.arguments

'{"intent": "app", "keywords": ["汽车故障", "轮胎"], "search": true}'

In [60]:
class Text(BaseModel):
    """抽取句子中的的单词，进行文本分词"""
    keyword: List[str] = Field(description="单词")
result = ExtractionAgent(model_name = "qwen-plus").call('小强是小王的好朋友。谢大脚是长贵的老公。', Text)
print(result)

keyword=['小强', '小王', '好朋友', '谢大脚', '长贵', '老公']


In [61]:
class Ticket(BaseModel):
    """根据用户提供的信息查询火车时刻"""
    date: str = Field(description="要查询的火车日期")
    departure: str = Field(description="出发城市或车站")
    destination: str = Field(description="要查询的火车日期")
result = ExtractionAgent(model_name = "qwen-plus").call("你能帮我查一下2024年1月1日从北京南站到上海的火车票吗？", Ticket)
print(result)

date='2024年1月1日' departure='北京南站' destination='上海'


In [62]:
class Text(BaseModel):
    """分析文本的情感"""
    sentiment: Literal["正向", "反向"] = Field(description="情感类型")
result = ExtractionAgent(model_name = "qwen-plus").call('我今天很开心。', Text)
print(result)

sentiment='正向'


In [66]:
class Text(BaseModel):
    """分析文本的情感"""
    sentiment: Literal["postive", "negative"] = Field(description="情感类型")
result = ExtractionAgent(model_name = "qwen-plus").call('我今天很开心。', Text)
print(result)

sentiment='postive'


In [71]:
class Text(BaseModel):
    """抽取实体"""
    person: List[str] = Field(description="人名")
    location: List[str] = Field(description="地名")
result = ExtractionAgent(model_name = "qwen-plus").call('今天我和徐也也去海淀吃饭，强哥也去了。', Text)
print(result)

person=['徐也也', '强哥'] location=['海淀']


In [72]:
class Text(BaseModel):
    """抽取句子中所有实体之间的关系"""
    source_person: List[str] = Field(description="原始实体")
    target_person: List[str] = Field(description="目标实体")
    relationship: List[Literal["朋友", "亲人", "同事"]] = Field(description="待选关系")
result = ExtractionAgent(model_name = "qwen-plus").call('小强是小王的好朋友。谢大脚是长贵的老公。', Text)
print(result)

source_person=['小强'] target_person=['小王'] relationship=['朋友']


In [76]:
class Text(BaseModel):
    """文本问答内容解析"""
    search: bool = Field(description="是否需要搜索")
    keywords: List[str] = Field(description="关键词")
    intent: Literal["查询客服问题", "查询产品问题", "查询系统问题", "其他"] = Field(description="意图")
result = ExtractionAgent(model_name = "qwen-plus").call('汽车发动和轮胎出故障了，如何处理？', Text)
print(result)


search=True keywords=['汽车发动', '轮胎故障'] intent='查询产品问题'


In [77]:
class Text(BaseModel):
    """文本问答内容解析"""
    time: List[str] = Field(description="时间")
    particate: List[str] = Field(description="选手")
    competition: List[str] = Field(description="赛事名称")
result = ExtractionAgent(model_name = "qwen-plus").call('2月8日上午北京冬奥会自由式滑雪女子大跳台决赛中中国选手谷爱凌以188.25分获得金牌！2022语言与智能技术竞赛由中国中文信息学会和中国计算机学会联合主办。', Text)
print(result)

time=['2月8日上午', '2022'] particate=['谷爱凌', '中国中文信息学会', '中国计算机学会'] competition=['北京冬奥会自由式滑雪女子大跳台决赛', '2022语言与智能技术竞赛']


In [89]:
from typing import Optional
class Text(BaseModel):
    """翻译请求解析器"""
    source_text: str = Field(description="需要翻译的原文")
    source_language: Optional[str] = Field(description="源语言，如不指定则自动检测")
    target_language: Optional[str] = Field(description="目标语言，从用户的提问中推测")
    result:str = Field(description="翻译的结果，用推测出来的目标语言")
    
result = ExtractionAgent(model_name = "qwen-plus").call('请翻译:China is a beautiful place', Text)
print(result)

result = ExtractionAgent(model_name = "qwen-plus").call('Please Translate:中国是一个美丽的地方', Text)
print(result)

source_text='China is a beautiful place' source_language='en' target_language='zh' result='中国是一个美丽的地方'
source_text='中国是一个美丽的地方' source_language='zh' target_language='en' result='China is a beautiful place.'


In [ ]:
原始语种、目标语种，待翻译的文本